In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
insurance_claims = pd.read_parquet("../data/clean_insurance_claims.parquet")

In [ ]:
# roughly 25:75 split, Y:N
insurance_claims["fraud_reported"].value_counts()

In [ ]:
# remove columns that are not useful for analysis or modelling
insurance_claims = insurance_claims.drop(columns = ["policy_number", "insured_zip", "incident_location"])

In [ ]:
# split data into numeric, categorical, and dates 
numeric_cols = insurance_claims.select_dtypes(include = ["int64", "float64"])
categorical_cols = insurance_claims.select_dtypes(include = ["object"])
datetime_cols = insurance_claims.select_dtypes(include = ["datetime64[ns]"])

numeric_cols["fraud_reported"] = insurance_claims["fraud_reported"]
datetime_cols["fraud_reported"] = insurance_claims["fraud_reported"]

#### NUMERIC COLUMNS
1. The boxplots suggest that most features do not differentiate between fraud and non-fraud. However, noticeable differences are observed for `witnesses`, `total_claim_amount`, `injury_claim`, `property_claim`, and `vehicle_claim`.

2. The histograms show an overlap between fraud and non-fraud, identical to the boxplot observations. The five features from the boxplots show noticeable differences in distributions, with the claim features having a far lower frequency than the lower claims (close to 0).

3. The correlation matrix shows a strong positive correlation between `age` and `months_as_customer`, which is expected as the older customers have been insured longer. Strong correlations are observed between the claim features, which is expected given that the features are derived from one another. There is also a weak positive correlation between `incident_hour_of_the_day` and `number_of_vehicles_involved` with the claim features. There is no correlation with fraud, suggesting that fraud will require multiple features to detect.

4. The independent t-tests comparing fraud vs non-fraud claims show that most features did not show statistically significant differences in their means at $\alpha = 0.05$. Only the claim-related variables were statistically significant, suggesting they are important predictors of fraud.

In [ ]:
numeric_names = numeric_cols.select_dtypes(include = ["int64", "float64"]).columns
palette = {"Y": "green", "N": "red"}

fig, axes = plt.subplots(4, 4, figsize = (12, 12), constrained_layout = True)
axes = axes.flatten() 

for axe, column in zip(axes, numeric_names):
    sns.boxplot(data = numeric_cols, x = "fraud_reported", y = column, ax = axe, hue = "fraud_reported", palette = palette)
    axe.set_title(column)

plt.show()

In [ ]:
numeric_names = numeric_cols.select_dtypes(include = ["int64", "float64"]).columns

fig, axes = plt.subplots(4, 4, figsize = (12, 12), constrained_layout = True)
axes = axes.flatten() 

for axe, column in zip(axes, numeric_names):
    sns.histplot(data = numeric_cols, x = column, hue = "fraud_reported", ax = axe, kde = True, legend = False, palette = palette)
    axe.set_title(column)

plt.show()

In [ ]:
numeric_cols.groupby("fraud_reported")[["witnesses", "total_claim_amount", "injury_claim", "property_claim", "vehicle_claim"]].mean().T

In [ ]:
numeric_cols.groupby("fraud_reported")[["witnesses", "total_claim_amount", "injury_claim", "property_claim", "vehicle_claim"]].median().T

In [ ]:
numeric_cols["fraud_numeric"] = numeric_cols["fraud_reported"].map({"N": 0, "Y": 1})
numeric_corr = numeric_cols.select_dtypes(include = ["int64", "float64"]).corr()

plt.figure(figsize = (16, 10), constrained_layout = True)

sns.heatmap(numeric_corr, annot = True, fmt = ".2g", linewidths = 0.5, cmap = "coolwarm", vmin = -1)
plt.title("Correlation of Numeric Features")

plt.show()

numeric_cols = numeric_cols.drop(columns = ["fraud_numeric"])

In [ ]:
t_test_scores = {}

for column in numeric_cols:
    if column == "fraud_reported":
        continue
    fraud_column = numeric_cols[numeric_cols["fraud_reported"] == "Y"][column]
    nonfraud_column = numeric_cols[numeric_cols["fraud_reported"] == "N"][column]

    t_stat, p_value = stats.ttest_ind(fraud_column, nonfraud_column, equal_var = False)
    t_test_scores[column] = (t_stat, p_value, p_value < 0.01)

t_test_scores_df = pd.DataFrame.from_dict(t_test_scores, orient = "index", columns = ["t_stat", "p_value", "significant"])
t_test_scores_df